#  Early Downy Mildew  Detection 



In [ ]:
!pip install torch torchvision torchaudio opencv-python scikit-image scikit-learn matplotlib seaborn pandas grad-cam streamlit pillow

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import json
import random
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Dataset Loading & EDA

In [ ]:
DATASET_PATH = 'dataset/'
classes = ['healthy', 'downey mildew']
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

image_paths = []
labels = []

for cls in classes:
    class_dir = os.path.join(DATASET_PATH, cls)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(class_dir, img_name))
                labels.append(class_to_idx[cls])

sns.countplot(x=[classes[label] for label in labels])
plt.title('Dataset Class Distribution')
plt.show()

print(f"Total images: {len(image_paths)}")

## 2. Preprocessing Pipeline
Applied: Resizing, Background Removal (GrabCut), Color Space Transform, Sharpening, NLM Denoising, Normalization.

In [ ]:
def remove_background_grabcut(img_rgb):
    '''Remove background using OpenCV GrabCut algorithm.'''
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    mask = np.zeros(img_bgr.shape[:2], np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    h, w = img_bgr.shape[:2]
    margin = int(min(h, w) * 0.05)
    rect = (margin, margin, w - 2*margin, h - 2*margin)
    cv2.grabCut(img_bgr, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
    mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
    # Replace background with white
    white_bg = np.ones_like(img_rgb) * 255
    result = np.where(mask2[:, :, np.newaxis] == 0, white_bg, img_rgb)
    return result.astype(np.uint8)

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Step 1: Resize
    img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)

    # Step 2: Background Removal (GrabCut)
    img = remove_background_grabcut(img)

    # Step 3: Color Space Transform - CLAHE on LAB L-channel
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    img = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)

    # Step 4: Image Filtering (Sharpening)
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    img = cv2.filter2D(img, -1, kernel)

    # Step 5: Noise Reduction (NLM Denoising)
    img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)

    return img

# Offline preprocessing — save processed images to disk
PREPROCESSED_PATH = 'preprocessed/'
os.makedirs(PREPROCESSED_PATH, exist_ok=True)
for cls in classes:
    os.makedirs(os.path.join(PREPROCESSED_PATH, cls), exist_ok=True)

preprocessed_paths = []
print("Preprocessing dataset (this may take a few minutes)...")
for path, label in zip(image_paths, labels):
    img_name = os.path.basename(path)
    save_path = os.path.join(PREPROCESSED_PATH, classes[label], img_name)
    if not os.path.exists(save_path):
        processed_img = preprocess_image(path)
        if processed_img is not None:
            cv2.imwrite(save_path, cv2.cvtColor(processed_img, cv2.COLOR_RGB2BGR))
    preprocessed_paths.append(save_path)
print(f"Preprocessing complete! Total: {len(preprocessed_paths)} images.")

### Preprocessing Visualization
Before vs. After for one sample from each class.

In [ ]:
def show_preprocessing_steps(original_path):
    img_orig = cv2.cvtColor(cv2.imread(original_path), cv2.COLOR_BGR2RGB)
    img_orig_resized = cv2.resize(img_orig, (224, 224), interpolation=cv2.INTER_AREA)

    # Step 2: GrabCut background removal
    img_bg = remove_background_grabcut(img_orig_resized)

    # Step 3: CLAHE on LAB
    lab = cv2.cvtColor(img_bg, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_clahe = cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2RGB)

    # Step 4: Sharpening
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    img_sharp = cv2.filter2D(img_clahe, -1, kernel)

    # Step 5: NLM Denoising
    img_final = cv2.fastNlMeansDenoisingColored(img_sharp, None, 10, 10, 7, 21)

    steps = [img_orig_resized, img_bg, img_clahe, img_sharp, img_final]
    titles = ['Original (Resized)', 'BG Removal (GrabCut)', 'Color Transform (LAB+CLAHE)', 'Sharpened', 'Final (Denoised)']

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    for ax, img, title in zip(axes, steps, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.suptitle(f'Preprocessing Steps: {os.path.basename(original_path)}', fontsize=12)
    plt.tight_layout()
    plt.show()

# Show for one sample per class
for cls in classes:
    cls_paths = [p for p, l in zip(image_paths, labels) if classes[l] == cls]
    if cls_paths:
        print(f"\nClass: {cls.upper()}")
        show_preprocessing_steps(cls_paths[0])

## 3. Data Augmentation & Loaders

In [ ]:
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class VineyardDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self): return len(self.file_paths)
        
    def __getitem__(self, idx):
        img = cv2.imread(self.file_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

X_train, X_temp, y_train, y_temp = train_test_split(preprocessed_paths, labels, test_size=0.3, stratify=labels, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_dataset = VineyardDataset(X_train, y_train, transform=train_transforms)
val_dataset = VineyardDataset(X_val, y_val, transform=val_test_transforms)
test_dataset = VineyardDataset(X_test, y_test, transform=val_test_transforms)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 4. Model Architectures

In [ ]:
def get_resnet50():
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

def get_efficientnet_b0():
    model = models.efficientnet_b0(pretrained=True)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    return model

def get_mobilenet_v2():
    model = models.mobilenet_v2(pretrained=True)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    return model

## 5. Training Loop with Early Stopping

In [ ]:
def train_model(model, name, train_loader, val_loader, epochs=20, patience=7):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    best_val_loss = float('inf')
    best_model_wts = model.state_dict()
    early_stop_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        running_loss, corrects, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)
            
        train_loss = running_loss / total
        train_acc = corrects.double() / total
        
        model.eval()
        val_loss, val_corrects, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_total += inputs.size(0)
                
        val_loss = val_loss / val_total
        val_acc = val_corrects.double() / val_total
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc.item())
        history['val_acc'].append(val_acc.item())
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = model.state_dict()
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print("Early stopping triggered!")
                break
                
    model.load_state_dict(best_model_wts)
    return model, history

## 6. Train Models

In [ ]:
print("Training ResNet50...")
resnet_model = get_resnet50()
resnet_model, resnet_hist = train_model(resnet_model, "ResNet50", train_loader, val_loader)

In [ ]:
print("Training EfficientNetB0...")
effnet_model = get_efficientnet_b0()
effnet_model, effnet_hist = train_model(effnet_model, "EfficientNetB0", train_loader, val_loader)

In [ ]:
print("Training MobileNetV2...")
mobilenet_model = get_mobilenet_v2()
mobilenet_model, mobilenet_hist = train_model(mobilenet_model, "MobileNetV2", train_loader, val_loader)

## 7. Comprehensive Evaluation — All Models

All metrics computed on the **held-out test set** for ResNet50, EfficientNetB0, and MobileNetV2.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    matthews_corrcoef, cohen_kappa_score, log_loss
)
import pandas as pd
import time

os.makedirs('results', exist_ok=True)

def full_evaluate(model, model_name, loader, class_names):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    t_start = time.time()
    with torch.no_grad():
        for inputs, labels_batch in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.numpy())
            all_probs.extend(probs.cpu().numpy())
    inference_time = (time.time() - t_start) / len(all_labels) * 1000  # ms per image

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    acc       = accuracy_score(all_labels, all_preds)
    prec      = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec       = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    f1_macro  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    mcc       = matthews_corrcoef(all_labels, all_preds)
    kappa     = cohen_kappa_score(all_labels, all_preds)
    logloss   = log_loss(all_labels, all_probs)
    roc_auc   = roc_auc_score(all_labels, all_probs[:, 1])
    avg_prec  = average_precision_score(all_labels, all_probs[:, 1])

    # ── 1. Classification Report ──────────────────────────────
    print(f"{'='*60}")
    print(f"  MODEL: {model_name}")
    print(f"{'='*60}")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # ── 2. Confusion Matrix ───────────────────────────────────
    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 14})
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=13)
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'results/{model_name}_confusion_matrix.png', dpi=150)
    plt.show()

    # ── 3. ROC Curve ─────────────────────────────────────────
    fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
    roc_auc_val = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC AUC = {roc_auc_val:.4f}')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve — {model_name}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(f'results/{model_name}_roc_curve.png', dpi=150)
    plt.show()

    # ── 4. Precision-Recall Curve ─────────────────────────────
    prec_curve, rec_curve, _ = precision_recall_curve(all_labels, all_probs[:, 1])
    plt.figure(figsize=(6, 5))
    plt.plot(rec_curve, prec_curve, color='steelblue', lw=2, label=f'AP = {avg_prec:.4f}')
    plt.xlabel('Recall'); plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve — {model_name}')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(f'results/{model_name}_pr_curve.png', dpi=150)
    plt.show()

    metrics = {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision (W)': round(prec, 4),
        'Recall (W)': round(rec, 4),
        'F1 (Weighted)': round(f1, 4),
        'F1 (Macro)': round(f1_macro, 4),
        'ROC-AUC': round(roc_auc_val, 4),
        'Avg Precision': round(avg_prec, 4),
        'Sensitivity': round(sensitivity, 4),
        'Specificity': round(specificity, 4),
        'MCC': round(mcc, 4),
        'Cohen Kappa': round(kappa, 4),
        'Log Loss': round(logloss, 4),
        'Inf Time (ms)': round(inference_time, 2)
    }
    return metrics

In [ ]:
all_metrics = []
models_dict = {
    "ResNet50":       resnet_model,
    "EfficientNetB0": effnet_model,
    "MobileNetV2":    mobilenet_model
}

for name, mdl in models_dict.items():
    m = full_evaluate(mdl, name, test_loader, classes)
    all_metrics.append(m)

### Training Curves — One Plot Per Model

In [ ]:
histories = {
    "ResNet50":       resnet_hist,
    "EfficientNetB0": effnet_hist,
    "MobileNetV2":    mobilenet_hist
}
colors = {"ResNet50": "steelblue", "EfficientNetB0": "darkorange", "MobileNetV2": "seagreen"}

for name, hist in histories.items():
    epochs = range(1, len(hist['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss plot
    axes[0].plot(epochs, hist['train_loss'], label='Train Loss', color=colors[name], linestyle='--', linewidth=2)
    axes[0].plot(epochs, hist['val_loss'],   label='Val Loss',   color=colors[name], linewidth=2)
    axes[0].set_title(f'{name} — Loss', fontsize=12)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Accuracy plot
    axes[1].plot(epochs, hist['train_acc'], label='Train Acc', color=colors[name], linestyle='--', linewidth=2)
    axes[1].plot(epochs, hist['val_acc'],   label='Val Acc',   color=colors[name], linewidth=2)
    axes[1].set_title(f'{name} — Accuracy', fontsize=12)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'Training vs Validation — {name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'results/{name}_training_curves.png', dpi=150)
    plt.show()
    print(f"Saved: results/{name}_training_curves.png")

### Model Comparison Summary

In [ ]:
df = pd.DataFrame(all_metrics).set_index('Model')
print("\n===== FULL MODEL COMPARISON TABLE =====")
print(df.to_string())

# Save to CSV for research paper
df.to_csv('results/model_comparison.csv')
print("\nSaved to results/model_comparison.csv")

# Bar chart comparing key metrics
key_metrics = ['Accuracy', 'F1 (Weighted)', 'ROC-AUC', 'MCC', 'Sensitivity', 'Specificity']
df_plot = df[key_metrics]
ax = df_plot.plot(kind='bar', figsize=(14, 6), rot=0, colormap='Set2', edgecolor='black', linewidth=0.5)
ax.set_title('Model Comparison — Key Metrics', fontsize=14)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=7, padding=2)
plt.tight_layout()
plt.savefig('results/model_comparison_chart.png', dpi=150)
plt.show()

# Best model selection
best_model_name = df['Accuracy'].idxmax()
best_model      = models_dict[best_model_name]
best_acc        = df.loc[best_model_name, 'Accuracy']
print(f"\nBest Model: {best_model_name}  |  Test Accuracy: {best_acc:.4f}")

torch.save(best_model.state_dict(), 'best_model.pth')
info = {"architecture": best_model_name, "num_classes": 2, "class_names": classes, "test_accuracy": best_acc}
with open('best_model_info.json', 'w') as f:
    json.dump(info, f)
print("Saved best_model.pth and best_model_info.json")

## 8. Explainable AI (Grad-CAM)
Grad-CAM applied on the **best model** for both correct and misclassified test samples.

In [ ]:
def get_target_layer(model, arch_name):
    if arch_name == "ResNet50":       return [model.layer4[-1]]
    elif arch_name == "EfficientNetB0": return [model.features[-1]]
    elif arch_name == "MobileNetV2":    return [model.features[-1]]

def apply_gradcam(model, architecture_name, image_path, true_label=None):
    model.eval()
    img = cv2.resize(cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB), (224, 224))
    rgb_img = img / 255.0
    input_tensor = val_test_transforms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(input_tensor)
        pred_idx = torch.argmax(out, dim=1).item()
        conf = torch.softmax(out, dim=1)[0][pred_idx].item() * 100

    target_layers = get_target_layer(model, architecture_name)
    cam = GradCAM(model=model, target_layers=target_layers)
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_idx)])[0, :]
    visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img); axes[0].set_title('Original Image'); axes[0].axis('off')
    status = ''
    if true_label is not None:
        status = 'CORRECT' if pred_idx == true_label else 'WRONG'
    axes[1].imshow(visualization)
    axes[1].set_title(f'Grad-CAM | Pred: {classes[pred_idx]} ({conf:.1f}%) {status}')
    axes[1].axis('off')
    plt.suptitle(f'Grad-CAM — {architecture_name}', fontsize=12)
    plt.tight_layout()
    plt.show()

# Show Grad-CAM for 3 test samples using best model
print(f"Grad-CAM on Best Model: {best_model_name}")
for i in range(min(3, len(X_test))):
    apply_gradcam(best_model, best_model_name, X_test[i], true_label=y_test[i])

## 9. 🏆 Final Cross-Model Comparison — Traditional ML vs CNN

Compares all models: Traditional (XGBoost/RF with HOG+LBP+Color), VGG16-feature ensembles, and fine-tuned CNNs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs('results', exist_ok=True)

# ── Load CNN results (already computed in Section 7)
cnn_df = pd.DataFrame(all_metrics)  # list built during full_evaluate calls
cnn_df['Model Type'] = 'Deep CNN (Fine-tuned)'

# ── Load Traditional model results (from traditional_features_ensemble_baseline.ipynb)
trad_csv = 'results/traditional_features/traditional_results.csv'

if os.path.exists(trad_csv):
    trad_df = pd.read_csv(trad_csv)
    trad_df['Model Type'] = 'Traditional ML (HOG+LBP+Color)'
    # Align columns: trad CSV uses same schema now
    combined = pd.concat([trad_df, cnn_df], ignore_index=True)
else:
    print("⚠️  Traditional results not found. Run traditional_features_ensemble_baseline.ipynb first.")
    print("    Showing CNN models only.")
    combined = cnn_df

combined = combined.set_index('Model')

print("\n" + "="*80)
print("  FULL CROSS-MODEL COMPARISON — Traditional ML vs Fine-tuned CNN")
print("="*80)
key_cols = ['Accuracy', 'Precision (W)', 'Recall (W)', 'F1 (Weighted)', 'ROC-AUC', 'MCC', 'Sensitivity', 'Specificity']
available = [c for c in key_cols if c in combined.columns]
print(combined[available].to_string())

# Save combined CSV
combined.to_csv('results/full_cross_model_comparison.csv')
print("\nSaved to results/full_cross_model_comparison.csv")

# ── Grouped Bar Chart ──────────────────────────────────────────────────────
plot_metrics = [c for c in ['Accuracy', 'F1 (Weighted)', 'ROC-AUC', 'MCC', 'Sensitivity', 'Specificity'] if c in combined.columns]
df_plot = combined[plot_metrics].reset_index()

colors_map = {
    'Traditional ML (HOG+LBP+Color)': '#E07B54',
    'Deep CNN (Fine-tuned)': '#4A90D9',
}

n_models  = len(df_plot)
n_metrics = len(plot_metrics)
x = np.arange(n_models)
bar_w = 0.13

fig, ax = plt.subplots(figsize=(16, 7))
palette = plt.cm.get_cmap('tab10', n_metrics)
for j, metric in enumerate(plot_metrics):
    bars = ax.bar(x + j * bar_w, df_plot[metric], width=bar_w,
                  label=metric, color=palette(j), edgecolor='black', linewidth=0.4)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=6, rotation=90)

ax.set_xticks(x + bar_w * (n_metrics / 2 - 0.5))
ax.set_xticklabels(df_plot['Model'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.18)
ax.set_title('Cross-Model Comparison — Traditional ML vs Fine-tuned CNN (Downy Mildew Detection)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.axhline(0.95, color='grey', linestyle='--', linewidth=0.8, alpha=0.6, label='0.95 reference')
plt.tight_layout()
plt.savefig('results/cross_model_comparison_chart.png', dpi=150)
plt.show()
print("Saved results/cross_model_comparison_chart.png")

# ── Winner summary
best_name = combined['Accuracy'].idxmax()
best_acc  = combined.loc[best_name, 'Accuracy']
best_f1   = combined.loc[best_name, 'F1 (Weighted)']
print(f"\n[BEST] Best Overall Model: {best_name}")
print(f"   Accuracy: {best_acc:.4f}  |  F1 (Weighted): {best_f1:.4f}")
